# Jurnal Keuangan Money Mind - Receipt OCR & Total Amount Classification
Dokumen ini memuat permodelan Machine Learning untuk mendeteksi nominal total belanja dari gambar struk secara lokal menggunakan **EasyOCR** (Deep Learning OCR) dan **Decision Tree Classifier** (Supervised Machine Learning).

## 1. Instalasi & Import Library
Kita memerlukan `easyocr` untuk mendeteksi teks pada gambar, dan `scikit-learn` untuk membuat model Decision Tree.

In [ ]:
# !pip install easyocr scikit-learn joblib pandas numpy

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib

## 2. Pembuatan Data Latih Sintetis (Synthetic Dataset)
Untuk melatih model Decision Tree kita, kita akan membuat simulasi data baris struk belanja. Setiap struk terdiri dari beberapa baris teks. Satu baris di antaranya adalah baris "Total Belanja" (Label = 1), sedangkan baris lainnya bukan (Label = 0).

Kita akan mengekstrak 3 fitur utama dari setiap baris:
1. `line_pos_ratio`: Posisi baris dibagi total baris (nilai 0.0 - 1.0). Baris total biasanya berada di bagian bawah (nilai mendekati 1.0).
2. `has_total_keyword`: Apakah baris mengandung kata kunci seperti 'total', 'jumlah', 'grand', 'bayar', 'rp' (1 jika ya, 0 jika tidak).
3. `has_number`: Apakah baris mengandung angka nominal (1 jika ya, 0 jika tidak).

In [ ]:
def generate_synthetic_receipt_data(n_receipts=1000):
    data = []
    
    for r_id in range(n_receipts):
        n_lines = np.random.randint(8, 22) # Setiap struk memiliki 8-22 baris
        total_line_idx = np.random.randint(int(n_lines * 0.6), n_lines - 1) # Baris total biasanya di 60%-90% bagian bawah
        
        for idx in range(n_lines):
            line_pos_ratio = idx / (n_lines - 1) if n_lines > 1 else 0.0
            
            # Tentukan apakah ini baris total belanja
            is_total = 1 if idx == total_line_idx else 0
            
            if is_total:
                has_total_keyword = 1 if np.random.rand() > 0.1 else 0
                has_number = 1
            else:
                # Baris biasa
                if line_pos_ratio < 0.5:
                    has_total_keyword = 1 if np.random.rand() > 0.98 else 0
                else:
                    has_total_keyword = 1 if np.random.rand() > 0.85 else 0
                has_number = 1 if np.random.rand() > 0.4 else 0
                
            data.append({
                'line_pos_ratio': line_pos_ratio,
                'has_total_keyword': has_total_keyword,
                'has_number': has_number,
                'label': is_total
            })
            
    return pd.DataFrame(data)

In [ ]:
df = generate_synthetic_receipt_data(1000)
print(f"Total baris data: {len(df)}")
df.head(15)

## 3. Training Model Decision Tree
Kita pisahkan data menjadi fitur (X) dan target label (y), kemudian bagi menjadi data training dan data testing.

In [ ]:
feature_cols = ['line_pos_ratio', 'has_total_keyword', 'has_number']
X = df[feature_cols]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inisialisasi Decision Tree Classifier dengan class_weight='balanced'
clf = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

## 4. Evaluasi Model
Kita uji performa model menggunakan data testing.

In [ ]:
y_pred = clf.predict(X_test)

print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

## 5. Menyimpan Model
Kita simpan model Decision Tree ini ke file `.joblib` agar bisa langsung digunakan oleh bot Telegram di backend Flask.

In [ ]:
model_path = 'backend/receipt_total_classifier.joblib'
joblib.dump(clf, model_path)
print(f"Model berhasil disimpan di: {model_path}")

## 6. Uji Coba Integrasi Pipeline OCR & Decision Tree (Local Simulation)
Berikut adalah simulasi bagaimana EasyOCR mengekstrak teks, dan model Decision Tree kita memprediksi baris total belanja secara lokal.

In [ ]:
# Simulasi teks hasil deteksi EasyOCR dari sebuah gambar struk belanja
ocr_output_simulation = [
    "WARUNG MAKAN KAMPUS",
    "Jl. Raya Perjuangan No. 10",
    "============================",
    "1x Nasi Goreng Spesial  18.000",
    "1x Es Teh Manis          5.000",
    "----------------------------",
    "SUBTOTAL                23.000",
    "Pajak 10%                2.300",
    "GRAND TOTAL             25.300",
    "TUNAI                   50.000",
    "KEMBALI                 24.700",
    "============================",
    "Terima Kasih Atas Kunjungan Anda"
]

def test_prediction(lines):
    total_lines = len(lines)
    keywords = ['total', 'jumlah', 'grand', 'bayar', 'rp', 'subtotal', 'netto']
    
    rows = []
    for idx, line in enumerate(lines):
        line_lower = line.lower()
        line_pos_ratio = idx / (total_lines - 1) if total_lines > 1 else 0.0
        has_total_keyword = 1 if any(kw in line_lower for kw in keywords) else 0
        has_number = 1 if re.search(r'\d', line) else 0
        
        rows.append({
            'line_pos_ratio': line_pos_ratio,
            'has_total_keyword': has_total_keyword,
            'has_number': has_number
        })
        
    df_features = pd.DataFrame(rows)
    
    # Prediksi menggunakan model yang sudah di-load
    loaded_clf = joblib.load('backend/receipt_total_classifier.joblib')
    probs = loaded_clf.predict_proba(df_features)[:, 1]
    
    # Terapkan filter tambahan (hard constraints)
    for idx in range(total_lines):
        if df_features.loc[idx, 'has_number'] == 0:
            probs[idx] = 0.0
        if df_features.loc[idx, 'line_pos_ratio'] < 0.3:
            probs[idx] = 0.0
            
    best_idx = np.argmax(probs)
    print(f"Baris terpilih sebagai Total Belanja: '{lines[best_idx]}' (Index: {best_idx}, Probabilitas: {probs[best_idx]:.2f})")
    
    # Ekstrak angka nominal
    predicted_line = lines[best_idx]
    clean_line = predicted_line.replace(',', '').replace(' ', '')
    clean_line = re.sub(r'\.(\d{3})', r'\1', clean_line)
    clean_line = re.sub(r'\.(\d{2})$', '', clean_line)
    numbers = re.findall(r'\d+', clean_line)
    amount = int(numbers[-1]) if numbers else 0
    print(f"Nominal Total Belanja: Rp{amount:,}")
    
    # Ekstrak nama merchant
    merchant_name = ""
    for line in lines:
        if len(re.findall(r'[a-zA-Z]', line)) >= 3 and not re.search(r'==|--|__', line):
            merchant_name = line.strip()
            break
    print(f"Nama Toko/Merchant: {merchant_name}")

test_prediction(ocr_output_simulation)

## 7. Deteksi Struk Menggunakan YOLOv8 (Computer Vision)
Bagian ini menunjukkan cara melatih model **YOLOv8** secara terprogram menggunakan dataset yang di-host di **Roboflow** untuk mendeteksi letak struk belanja di dalam foto dan memotongnya (*cropping*) sebelum dikirim ke modul OCR.

### 7.1 Instalasi Library YOLOv8 & Roboflow

In [ ]:
# !pip install ultralytics roboflow

### 7.2 Mengunduh Dataset dari Roboflow
Ganti `'YOUR_ROBOFLOW_API_KEY'` dengan API Key akun Roboflow Anda untuk mengunduh dataset secara otomatis dalam format YOLOv8.

In [ ]:
from roboflow import Roboflow

# Masukkan API Key Anda di sini
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("mercury-nxonz").project("deteksi-struk-belanja-oc8mm")
version = project.version(5)
dataset = version.download("yolov8")

print(f"Dataset berhasil diunduh dan disimpan di: {dataset.location}")

### 7.3 Melatih Model YOLOv8 secara Lokal
Kita akan memuat model pre-trained paling ringan yaitu **YOLOv8n (YOLOv8 Nano)** agar proses training lebih cepat, lalu melatihnya menggunakan dataset yang diunduh.

In [ ]:
from ultralytics import YOLO

# 1. Load model YOLOv8 Nano pre-trained
model = YOLO('yolov8n.pt')

# 2. Latih model
# - epochs=10 (demo singkat, untuk hasil optimal gunakan 50-100)
# - imgsz=640 (ukuran resolusi gambar input)
# - device='cpu' (ganti dengan device=0 jika Anda memiliki GPU NVIDIA CUDA)
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=10,
    imgsz=640,
    device='cpu'
)

print("Training Selesai! Model terbaik disimpan di: runs/detect/train/weights/best.pt")

### 7.4 Deteksi & Pemotongan Struk (Cropping Pipeline)
Setelah model dilatih, kita bisa menggunakannya untuk mendeteksi posisi struk di dalam gambar, memotongnya secara otomatis, lalu mengirimkannya ke modul pembaca nominal (EasyOCR).

In [ ]:
from PIL import Image

def crop_receipt_with_yolo(image_path, model_path='runs/detect/train/weights/best.pt'):
    # Load model hasil training
    yolo_model = YOLO(model_path)
    
    # Jalankan inferensi
    results = yolo_model(image_path)
    
    # Ambil bounding box objek pertama yang terdeteksi
    boxes = results[0].boxes
    if len(boxes) > 0:
        # Ambil koordinat [x_min, y_min, x_max, y_max]
        x1, y1, x2, y2 = map(int, boxes[0].xyxy[0])
        
        # Buka gambar asli dan lakukan cropping
        original_img = Image.open(image_path)
        cropped_img = original_img.crop((x1, y1, x2, y2))
        
        print(f"Struk berhasil dideteksi pada koordinat: [{x1}, {y1}, {x2}, {y2}]")
        return cropped_img
    else:
        print("Struk tidak terdeteksi oleh model YOLO. Menggunakan gambar asli.")
        return Image.open(image_path)

# Contoh penggunaan (simulasi):
# cropped = crop_receipt_with_yolo('test_struk.jpg')
# cropped.show() # Tampilkan struk yang terpotong bersih
# ocr_results = reader.readtext(np.array(cropped)) # Masuk ke EasyOCR

### 7.5 Evaluasi Akurasi & Validasi Model YOLOv8
Untuk menguji akurasi model YOLOv8 yang sudah dilatih pada dataset validasi, kita dapat menjalankan proses validasi menggunakan fungsi `model.val()`. Fungsi ini akan memuat dataset validasi (berdasarkan file `data.yaml`) dan menghitung metrik performa utama seperti **Precision**, **Recall**, dan **mAP (mean Average Precision)**.

In [ ]:
import os
from ultralytics import YOLO

# Path ke model hasil training terbaik
model_path = 'runs/detect/train/weights/best.pt'

if os.path.exists(model_path):
    print(f"Memuat model YOLOv8 dari: {model_path}...")
    model = YOLO(model_path)
    
    print("Menjalankan validasi model (model.val()) pada dataset validasi...")
    # Menjalankan validasi pada split validasi
    metrics = model.val(split='val')
    
    print("\n=== METRIK AKURASI YOLOV8 ===")
    print(f"mAP 50-95  : {metrics.box.map:.4f} (Rata-rata presisi pada IoU 50% hingga 95%)")
    print(f"mAP 50     : {metrics.box.map50:.4f} (Rata-rata presisi pada IoU 50%)")
    print(f"Precision  : {metrics.box.mp:.4f} (Rasio prediksi benar terhadap total prediksi)")
    print(f"Recall     : {metrics.box.mr:.4f} (Rasio struk yang terdeteksi terhadap total struk asli)")
else:
    print(f"⚠️ Berkas model '{model_path}' belum ditemukan.")
    print("Silakan jalankan training model YOLOv8 (Cell 21) terlebih dahulu sampai selesai untuk menghasilkan file 'best.pt'.")
    print("\nMetode evaluasi yang digunakan setelah training selesai:")
    print("""python
model = YOLO('runs/detect/train/weights/best.pt')
metrics = model.val(split='val')
print('mAP 50:', metrics.box.map50)
""")